# Training a two-layer GCN by hand

Ninety air-quality sensors are scattered over a field, and each one sits in one of three
pollution zones. A sensor's own readings are noisy, so its zone is hard to tell from them alone,
but sensors close to each other usually share a zone. Only fifteen sensors have been checked by
hand. You train a two-layer GCN on those fifteen, with the forward pass, the loss and the
gradients written in NumPy, and compare it with the same network when it cannot see the graph.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

field = np.random.default_rng(12)
n = 90
pos = field.random((n, 2))                              # sensor positions in a unit square
y = np.digitize(pos[:, 0] + pos[:, 1], [0.75, 1.25])    # zone 0, 1 or 2: three bands across the field
dist = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(axis=-1))
A = ((dist < 0.17) & (dist > 0)).astype(float)          # sensors closer than 0.17 are linked

# Five readings per sensor. The first three carry a weak signal of the zone, the last two are noise.
X = np.hstack([np.eye(3)[y], np.zeros((n, 2))]) + field.normal(scale=0.9, size=(n, 5))

# Five sensors of each zone were checked by hand. The other 75 are the test set.
mask = np.zeros(n, dtype=bool)
for c in range(3):
    mask[np.flatnonzero(y == c)[:5]] = True

def normalized_adjacency(A):
    A_hat = A + np.eye(len(A))
    d = A_hat.sum(axis=1)
    return A_hat / np.sqrt(np.outer(d, d))

S = normalized_adjacency(A)

def init_weights(seed, d_in=5, hidden=16, classes=3):
    r = np.random.default_rng(seed)
    W1 = r.normal(scale=np.sqrt(2 / d_in), size=(d_in, hidden))
    W2 = r.normal(scale=np.sqrt(1 / hidden), size=(hidden, classes))
    return W1, W2

def accuracy(Z, y, rows):
    """The share of the selected rows whose largest logit sits at their label."""
    return np.mean(Z[rows].argmax(axis=1) == y[rows])

print(f"{int(A.sum() / 2)} links, {mask.sum()} labeled sensors, zone sizes {np.bincount(y)}")

## Part 1 (coding task): The forward pass and the loss

The network has two GCN layers, a ReLU between them and none after the second:

$$
Z = S\,\operatorname{ReLU}(S X W_1)\,W_2 ,
$$

so $Z$ holds three logits per sensor. Write `forward(S, X, W1, W2)`, which returns `Z` and a
dictionary `cache` holding whatever your gradients in Part 2 will need. Then write
`masked_cross_entropy(Z, y, mask)`, the softmax cross-entropy of the logits against the labels,
averaged over the sensors where `mask` is True. Every sensor takes part in the forward pass, since
its readings reach its neighbors, but only the labeled ones enter the loss. The loss has to stay
finite, and right, when the logits run into the thousands.

In [ ]:
def forward(S, X, W1, W2):
    """The logits S relu(S X W1) W2, and a cache for the gradients."""
    # TODO
    raise NotImplementedError

def masked_cross_entropy(Z, y, mask):
    """The mean softmax cross-entropy over the rows where mask is True."""
    # TODO
    raise NotImplementedError

In [ ]:
W1, W2 = init_weights(0)
Z, cache = forward(S, X, W1, W2)
assert np.shape(Z) == (n, 3), f"expected logits of shape ({n}, 3), got {np.shape(Z)}"
loss = masked_cross_entropy(Z, y, mask)
y_other = y.copy()
y_other[~mask] = (y[~mask] + 1) % 3
assert np.isclose(masked_cross_entropy(Z, y_other, mask), loss), (
    "the loss changed when only the labels of unlabeled sensors changed: average over the masked rows only")
assert np.isclose(loss, 1.118810, atol=1e-6), (
    f"the loss at the starting weights should be 1.118810, got {loss:.6f}: "
    "check both multiplications by S, the ReLU between the layers, and that it averages over the 15 labeled rows")
# A confident wrong answer and an undecided one: the losses are 2000 and log 3.
big = masked_cross_entropy(np.array([[1000.0, 0.0, -1000.0], [0.0, 0.0, 0.0]]), np.array([2, 1]),
                           np.array([True, True]))
assert np.isfinite(big), f"with logits of 1000 and -1000 the loss is {big}"
assert np.isclose(big, (2000 + np.log(3)) / 2), f"the two rows should give a loss of {(2000 + np.log(3)) / 2:.4f}, got {big:.4f}"
print("Part 1 passed")

## Part 2 (coding task): The gradients

Write `gradients(S, X, W1, W2, y, mask)`, which returns `(dW1, dW2)`, the gradients of
`masked_cross_entropy(forward(S, X, W1, W2)[0], y, mask)` with respect to `W1` and `W2`, each
shaped like its weight matrix. The check compares them with central differences of your Part 1
functions.

In [ ]:
def gradients(S, X, W1, W2, y, mask):
    """(dW1, dW2): the gradients of the masked loss with respect to W1 and W2."""
    # TODO
    raise NotImplementedError

In [ ]:
def numeric_grad(f, M, h=1e-6):
    """Central differences of f() in every entry of M. M is nudged in place and restored."""
    G = np.zeros_like(M)
    for idx in np.ndindex(M.shape):
        old = M[idx]
        M[idx] = old + h
        up = f()
        M[idx] = old - h
        down = f()
        M[idx] = old
        G[idx] = (up - down) / (2 * h)
    return G

W1, W2 = init_weights(1)
dW1, dW2 = gradients(S, X, W1, W2, y, mask)
assert np.shape(dW1) == W1.shape, f"dW1 should have the shape of W1, {W1.shape}, got {np.shape(dW1)}"
assert np.shape(dW2) == W2.shape, f"dW2 should have the shape of W2, {W2.shape}, got {np.shape(dW2)}"
loss_now = lambda: masked_cross_entropy(forward(S, X, W1, W2)[0], y, mask)
assert np.allclose(dW2, numeric_grad(loss_now, W2), atol=1e-6), (
    "dW2 does not match the numeric gradient: look at the gradient at the logits, which rows it "
    "covers and what it is divided by")
assert np.allclose(dW1, numeric_grad(loss_now, W1), atol=1e-6), (
    "dW2 is right but dW1 is not: look at the step back through S, W2 and the ReLU")
print("Part 2 passed")

## Part 3 (coding task): Training

Write `train(S, X, y, mask, steps=200, lr=0.2, seed=0)`. It starts from `init_weights(seed)`,
takes `steps` steps of full-batch gradient descent on the masked loss with learning rate `lr`, and
returns the trained `(W1, W2)`. The check then trains the network twice: once on the sensor graph,
and once with $S$ replaced by the identity, which turns the GCN into a two-layer MLP that sees
each sensor's own readings only.

In [ ]:
def train(S, X, y, mask, steps=200, lr=0.2, seed=0):
    """Full-batch gradient descent from init_weights(seed). Returns (W1, W2)."""
    # TODO
    raise NotImplementedError

In [ ]:
W1_0, W2_0 = init_weights(0)
none = train(S, X, y, mask, steps=0)
assert np.allclose(none[0], W1_0) and np.allclose(none[1], W2_0), (
    "with steps=0 train should return the starting weights from init_weights(seed)")
g1, g2 = gradients(S, X, W1_0, W2_0, y, mask)
one = train(S, X, y, mask, steps=1, lr=0.3)
assert np.allclose(one[0], W1_0 - 0.3 * g1) and np.allclose(one[1], W2_0 - 0.3 * g2), (
    "one step with lr=0.3 should move each weight by -0.3 times its gradient")

W1, W2 = train(S, X, y, mask)
Z = forward(S, X, W1, W2)[0]
train_loss = masked_cross_entropy(Z, y, mask)
assert train_loss < 0.05, f"after 200 steps the training loss should be below 0.05, got {train_loss:.4f}"
y_test_changed = y.copy()
y_test_changed[~mask] = (y[~mask] + 2) % 3
again = train(S, X, y_test_changed, mask)
assert np.allclose(again[0], W1) and np.allclose(again[1], W2), (
    "the trained weights changed when only the test labels changed: the test labels reach training")

gcn_acc = accuracy(Z, y, ~mask)
I = np.eye(n)
M1, M2 = train(I, X, y, mask)
mlp_acc = accuracy(forward(I, X, M1, M2)[0], y, ~mask)
print(f"test accuracy: GCN {gcn_acc:.3f}, MLP without the graph {mlp_acc:.3f}")
assert gcn_acc > 0.85, f"the GCN should reach above 0.85 on the 75 test sensors, got {gcn_acc:.3f}"
print("Part 3 passed")

## Solution

### Part 1: The forward pass and the loss

In [ ]:
def forward(S, X, W1, W2):
    """The logits S relu(S X W1) W2, and a cache for the gradients."""
    SX = S @ X
    P = SX @ W1                      # first layer before the ReLU
    H1 = np.maximum(P, 0)
    SH1 = S @ H1
    Z = SH1 @ W2
    return Z, {"SX": SX, "P": P, "SH1": SH1}

def log_softmax(Z):
    """The log of the softmax of each row, shifted by the row's largest logit so no exp overflows."""
    top = Z.max(axis=1, keepdims=True)
    return Z - top - np.log(np.exp(Z - top).sum(axis=1, keepdims=True))

def masked_cross_entropy(Z, y, mask):
    """The mean softmax cross-entropy over the rows where mask is True."""
    log_p = log_softmax(Z[mask])
    return -np.mean(log_p[np.arange(mask.sum()), y[mask]])

**Why it works.** The forward pass is the GCN layer twice, with $W = W_1$ and a ReLU, then $W = W_2$ and no
nonlinearity, so the logits of a sensor depend on the readings of every sensor within two links.
The cache keeps $SX$, the first layer before its ReLU and $S\operatorname{ReLU}(SXW_1)$, the
three arrays the gradients multiply by. The loss subtracts each row's largest logit before
exponentiating. That leaves the log-softmax unchanged, because the shift cancels between a logit and
the log of the row's total, and keeps every exponent at most 0. Without it, a logit above about
709 overflows `exp` to `inf` and the loss comes out `nan`. Taking the log of the softmax
probabilities fails the other way: a label whose logit sits about 745 below the row's largest gets
probability 0 in floating point, and its log is $-\infty$. Working with $z - \log\sum e^{z}$
avoids both. Indexing with `mask` first means the
unlabeled rows never reach the mean, which is what makes this semi-supervised: their labels play
no part, but their readings still flow through $S$.

### Part 2: The gradients

In [ ]:
def gradients(S, X, W1, W2, y, mask):
    """(dW1, dW2): the gradients of the masked loss with respect to W1 and W2."""
    Z, cache = forward(S, X, W1, W2)
    probs = np.exp(log_softmax(Z))
    G = (probs - np.eye(Z.shape[1])[y]) * mask[:, None] / mask.sum()   # dL/dZ: zero on unlabeled rows
    dW2 = cache["SH1"].T @ G
    dH1 = S.T @ G @ W2.T
    dP = dH1 * (cache["P"] > 0)                                     # the ReLU passes gradient where P > 0
    dW1 = cache["SX"].T @ dP
    return dW1, dW2

**Why it works.** Write $G = \partial L / \partial Z$. For softmax cross-entropy a row's gradient is its softmax
minus the one-hot label, and the loss averages over the $m = 15$ labeled rows, so a labeled row
is divided by $m$ and an unlabeled row is 0. Dividing by all 90 rows, or keeping the unlabeled
rows, gives the gradient of a different loss. With $Z = (SH_1)W_2$, the second weight gets
$(SH_1)^\top G$. Going back, $\partial L / \partial H_1 = S^\top G W_2^\top$. The transpose of
$S$ matters in general, although this $S$ is symmetric. The ReLU passes the gradient only where
the first layer was positive, and $W_1$ sits in $(SX)W_1$, so it gets $(SX)^\top$ times that.
Even unlabeled sensors carry gradient back to $W_1$: their rows of $G$ are zero, but they
neighbor labeled sensors, so $S^\top G$ is not.

### Part 3: Training

In [ ]:
def train(S, X, y, mask, steps=200, lr=0.2, seed=0):
    """Full-batch gradient descent from init_weights(seed). Returns (W1, W2)."""
    W1, W2 = init_weights(seed)
    for _ in range(steps):
        dW1, dW2 = gradients(S, X, W1, W2, y, mask)
        W1 = W1 - lr * dW1
        W2 = W2 - lr * dW2
    return W1, W2

**Why it works.** Each step recomputes both gradients at the current weights and moves against them, and nothing
in it reads a test label, since `gradients` zeroes every unlabeled row of $G$. Both networks fit
the fifteen labeled sensors almost perfectly, so the difference lies in the test sensors. The MLP
gets about 0.59 of them right. It sees five noisy readings per sensor, and fifteen examples teach
it the noise as much as the signal. The GCN gets about 0.89. Each multiplication by $S$ replaces
a sensor's readings with a weighted combination of itself and its linked sensors whose weights
are set by the degree normalization. Those sensors mostly share its zone, so the noise averages
down while the zone signal stays. Two layers average over
everything within two links. The eight test sensors it still gets wrong all lie within 0.12 of a
border between bands, closer than one link length (0.17), where a neighborhood reaches into the
next zone. About half of the test sensors (37 of 75) lie that close.